Groundwater | Flow Modeling Track

# Step 5: Model Calibration – From Observations to Parameters

> **The 10 steps at a glance:** 1-Goal → 2-Perceptual → 3-MODFLOW → 4-Build → **5-Calibrate** → 6-Validate → 7-Sensitivity → 8-Apply → 9-Document → 10-Communicate

**Story so far:** In Step 4 you built a MODFLOW 6 model of the Limmat Valley aquifer with a Voronoi (DISV) grid and uniform hydraulic conductivity. The model runs, but we haven’t checked whether it actually reproduces the groundwater levels we observe in the field. That’s what calibration does, and we’ll discover along the way that we need more data and more flexible parameterisation than we started with.

By default, Section 5 uses a **pre-computed PEST++ calibration** that we download for you — this keeps the required path fast and the rest of the notebook deterministic. If you want to run PEST++ locally on your machine, flip `RUN_PEST_LOCALLY = True` in Section 5.0 (adds about 10 minutes of compute).

| **Core content** | ~90-110 minutes |
|:--|:--|
| **Optional: PEST local run, exercises & advanced topics** | +10 minutes (PEST) + ~20 minutes (exercises) |

---
## Learning Objectives

By the end of this notebook, you will be able to:

1. **Assess** observation data coverage and identify spatial gaps in monitoring networks
2. **Evaluate** a pumping test using the Cooper-Jacob straight-line method
3. **Calculate** calibration metrics (ME, MAE, RMSE, R², NRMSE) and interpret their meaning
4. **Explain** how PEST++ with pilot points estimates spatially varying parameters
5. **Interpret** a calibrated hydraulic conductivity field and the role of prior information
6. **Verify** calibration quality through water balance checks and residual analysis

---
## Prerequisites

Before starting this notebook, you should have:
- **Completed [04f_model_implementation.ipynb](04f_model_implementation.ipynb)** — your model must run successfully
- Basic understanding of **Darcy’s Law** and the concept of hydraulic conductivity
- Familiarity with the **Limmat Valley case study** from the earlier notebooks

In [ ]:
# Setup
import sys
import os
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

import flopy
from flopy.mf6 import MFSimulation

from IPython.display import display, HTML

# Add support modules to path
sys.path.append(os.path.abspath('../../_SUPPORT/src'))
sys.path.append(os.path.abspath('../../_SUPPORT/src/scripts'))
sys.path.append(os.path.abspath('../../_SUPPORT/src/scripts/scripts_exercises'))

from env_utils import ensure_package
ensure_package('pyemu', 'pyemu==1.4.0')  # self-heal if the hub kernel lacks pyemu (05f-08f need it)

from data_utils import (
    download_named_file,
    get_default_data_folder,
)
from model_io_utils import load_base_simulation, load_model_boundary, format_budget_summary
from plot_utils import quick_model_plot, check_heads_above_surface

# Checkpoint utilities
from shared_functions import check_task_with_solution, create_multiple_choice

import pumping_test_utils as ptu
import calibration_utils as cu

# Plotting defaults
plt.rcParams['figure.figsize'] = (10, 7)
plt.rcParams['font.size'] = 12

## Introduction

Calibration is the process of adjusting model parameters so that simulated outputs match field observations as closely as possible. Without calibration, a model is simply a hypothesis. In Step 4 we assigned a uniform hydraulic conductivity of K = 200 m/d. But how do we know that value is correct? The answer is: we don’t know until we compare the model’s predictions to actual measurements. This is why we need calibration, to tests that hypothesis against reality.

This notebook follows a natural problem-solving story:

1. **Load the model** and see what observation data we have → we discover we have very few wells
2. **Run the uncalibrated model** and check the fit → it may look OK locally, but we have no data elsewhere
3. **Analyse a pumping test** to get an independent K estimate in a data-sparse area
4. **Calibrate with PEST++** using pilot points and the pumping-test K as prior information
5. **Assess the calibrated model** — does it make physical sense?

---

## 1 - Load Model and Observation Data

### 1.1 - Load model data

We start by loading the MODFLOW 6 model from Step 4 and copying it to a separate **calibration workspace** (so we don’t modify the original model files).

In [ ]:
# --- 1. Define paths ---
DATA_DIR = get_default_data_folder()

# The model from Notebook 4 lives here:
sim_name = 'limmat_valley'
nb4_workspace = os.path.join(DATA_DIR, 'notebook4_model')

# We'll work in a separate calibration workspace to keep the original model intact
calib_workspace = os.path.join(DATA_DIR, 'calibration')

# Always start fresh: copy the NB4 model to the calibration workspace
if not os.path.exists(os.path.join(nb4_workspace, 'mfsim.nam')):
    raise FileNotFoundError(
        f"Notebook 4 model not found at {nb4_workspace}\n"
        "Please run Notebook 4 first to generate the MODFLOW 6 model."
    )

if os.path.exists(calib_workspace):
    shutil.rmtree(calib_workspace)
shutil.copytree(nb4_workspace, calib_workspace)
print(f"Fresh calibration workspace created: {calib_workspace}")

workspace = calib_workspace

# --- 2. Load the MODFLOW 6 simulation ---
sim = load_base_simulation(workspace)
gwf = sim.get_model(sim_name)

modelgrid = gwf.modelgrid
modelgrid.set_coord_info(crs="EPSG:2056")

# Get idomain for masking
disv = gwf.get_package('DISV')
idomain = disv.idomain.array

# Clean up any WEL entries in inactive cells (can occur from NB4 sensitivity runs)
wel = gwf.get_package('WEL')
if wel is not None:
    wd = wel.stress_period_data.get_data(0)
    if wd is not None and len(wd) > 0:
        id_flat = idomain.flatten()
        keep = [r for r in wd if id_flat[r['cellid'][-1]] > 0]
        n_removed = len(wd) - len(keep)
        if n_removed > 0:
            import numpy.lib.recfunctions as rfn
            wel.stress_period_data.set_data(keep, key=0)
            print(f"Removed {n_removed} WEL entries in inactive cells.")

print(f"Grid type: {modelgrid.grid_type}")
print(f"Number of cells per layer: {modelgrid.ncpl}")
print(f"Active cells: {(idomain.flatten() > 0).sum()}")

# --- 3. Load model boundary for spatial filtering ---
boundary_path = download_named_file(name='model_boundary', data_type='gis')
boundary_gdf = gpd.read_file(boundary_path)

### 1.2 - Load observation data

#### 1.2.1 - Load AWEL observation data

The Canton of Zürich (AWEL) maintains a network of groundwater monitoring wells with long-term head measurements. Let's load this data and see which wells fall within our model domain.

In [ ]:
# --- 4. Load AWEL observation wells ---
# load_awel_observations handles:
#   - Reading the CSV time series
#   - Converting LV03 → LV95 coordinates
#   - Computing mean heads per well
#   - Filtering to wells within the model boundary
obs_gdf = cu.load_awel_observations(DATA_DIR, boundary_gdf, use_mean=True)

print(f"Observation wells within model domain: {len(obs_gdf)}")
print(cu.get_observation_summary(obs_gdf))
display(obs_gdf[['obs_id', 'x', 'y', 'head_m']])

#### 1.2.2 - Generate synthetic observations

Since the real AWEL wells are clustered at the eastern end of the aquifer, we supplement them with **synthetic observation points** to provide spatial coverage for the calibration demonstration.

The synthetic observations are sampled from a **reference model** with spatially varying K that reflects the geological heterogeneity of the alluvial aquifer. In the Limmat Valley, aquifer thickness increases from west (~10 m) to east (~50 m), and deeper sections tend to contain coarser gravels with higher K. The reference K field captures this thickness-dependent trend plus random spatial variability, producing head patterns more realistic than a uniform-K model.

In [ ]:
# --- 5. Generate synthetic observations ---
# We create a "reference" K field (conditioned on AWEL heads, with thickness-dependent
# trends and spatial noise), run the model with it, and sample synthetic head observations.
# This gives us something to calibrate against in data-sparse areas.

top = disv.top.array.flatten()
botm = disv.botm.array.flatten()

k_reference, k_conditioned, cond_info = cu.generate_conditioned_k_field(
    sim, gwf, modelgrid, idomain, top, botm,
    obs_gdf=obs_gdf, seed=42, noise_std=0.25,
    variogram_range=3000.0, anisotropy_angle=-30.0, anisotropy_scaling=3.0,
)

# Run with reference K, then restore uniform K=40
gwf.npf.k.set_data(k_reference)
sim.write_simulation()
success, _ = sim.run_simulation(silent=True)
if not success:
    raise RuntimeError("Reference model run failed — check the listing file.")
head_ref = gwf.output.head().get_data()
gwf.npf.k.set_data(200.0)

# Identify river cells and river polygons for spatial filtering
riv = gwf.get_package('RIV')
riv_data = riv.stress_period_data.get_data(0) if riv is not None else None
riv_cells = np.array([r[1] for r in riv_data]) if riv_data is not None else np.array([])

river_gis_path = os.path.join(os.path.dirname(boundary_path), 'AV_Gewasser_-OGD.gpkg')
river_all = gpd.read_file(river_gis_path)
river_gdf = river_all[
    river_all['GEWAESSERNAME'].isin(['Limmat', 'Sihl'])
    & river_all.intersects(boundary_gdf.geometry.iloc[0])
]

# Sample 5 synthetic observations south of the river, away from boundaries
synth_gdf = cu.generate_synthetic_observations(
    modelgrid, head_ref, idomain,
    n_points=5, noise_std=0.5, seed=42,
    exclude_cells=riv_cells, river_gdf=river_gdf,
    boundary_polygon=boundary_gdf.geometry.iloc[0],
    avoid_boundaries_m=100, exclude_north_of_river=True,
)

obs_gdf = cu.combine_observations(obs_gdf, synth_gdf)
print(cu.get_observation_summary(obs_gdf))

# --- 6. Display combined observation dataset ---
display(obs_gdf[['obs_id', 'x', 'y', 'head_m', 'is_synthetic', 'source']].reset_index(drop=True))

In [ ]:
# Checkpoint 1: How many total observation points do you have?
check_task_with_solution('task05_checkpoint_1')

### 1.2.3 - Visualise the observation network

The 4 real AWEL wells are clustered at the eastern end of the aquifer. Without the synthetic supplements, there would be no observation data in the central or downstream portions. Even with synthetic points, only 4 independent measurements constrain the model and the rest are artificial.

> **Think about it:** If we calibrate uniform K to match heads at a few nearby wells, how confident can we be in the model's predictions 2 km downstream?

In [ ]:
# --- 7. Map of observation wells on the model grid ---
fig = cu.plot_observation_network(
    obs_gdf,
    boundary_gdf=boundary_gdf,
    title='Observation Network — AWEL + Synthetic Wells',
)
plt.show()

---

## 2 - Initial Model Assessment

Let’s run the model with its current parameterisation (uniform K = 200 m/d from Notebook 4) and see how well it matches the observations.
To compare, we rely on our interpretation of the residual at observation points  (model result - observation value).

### 2.1 - How to read residuals

Understanding residual patterns is the key to intelligent calibration (it's the physical reasoning that PEST++ automates)
**Physical intuition:** Higher K → water flows more easily → heads drop. Lower K → water backs up → heads rise. This is Darcy's Law at work.

| Residual pattern | Meaning | Likely parameter issue |
|:--|:--|:--|
| **Positive** (sim > obs) | Simulated heads too high | K too **low** — water can't drain away fast enough |
| **Negative** (sim < obs) | Simulated heads too low | K too **high** — water drains too fast |
| **Spatial clustering** | Residuals of the same sign grouped together | Zone-specific K or recharge problem |
| **Random scatter** | No spatial pattern | Good fit — remaining error is noise |



In [ ]:
# Checkpoint: Interpret the residual pattern
create_multiple_choice('task05_checkpoint_5')

### 2.2 - Assessment results

In [ ]:
# --- 1. Run the model ---
sim.write_simulation()
success, _ = sim.run_simulation(silent=True)

if success:
    print("Model run completed successfully.")
else:
    print("Model run FAILED. Check the listing file for errors.")

# --- 2. Load simulated heads ---
head = gwf.output.head().get_data()
heads_flat = head.flatten() if head.ndim > 1 else head

print(f"Head array shape: {head.shape}")
valid_heads = heads_flat[heads_flat > -1e10]
print(f"Head range: {valid_heads.min():.2f} to {valid_heads.max():.2f} m a.s.l.")

# --- 3. Extract simulated heads at observation wells ---
sim_heads = cu.extract_heads_at_observations(head, obs_gdf, modelgrid)
obs_gdf['simulated_head'] = sim_heads

# Remove any wells that couldn't be matched to grid cells
valid_mask = ~np.isnan(sim_heads)
obs_valid = obs_gdf[valid_mask].copy()

metrics_initial = cu.calculate_calibration_metrics(
    obs_valid['head_m'].values, obs_valid['simulated_head'].values,
)
print(cu.format_metrics_table(metrics_initial))

# --- 4. Observed vs. simulated scatter plot ---
print("We can look at the difference between model and observation for each observation location.")
fig = cu.plot_calibration_scatter(
    obs_valid['head_m'].values,
    obs_valid['simulated_head'].values,
    obs_gdf=obs_valid,
    title='Initial (Uncalibrated) Model',
)
plt.show()

# --- 5. Residual map ---
residuals = obs_valid['simulated_head'].values - obs_valid['head_m'].values
fig = cu.plot_residual_map(
    obs_valid, residuals,
    boundary_gdf=boundary_gdf,
    title='Residual Map — Uncalibrated Model',
)
plt.show()

In [ ]:
# Checkpoint 2: What is the initial RMSE (m)?
check_task_with_solution('task05_checkpoint_2')

### 2.3 - Quick manual trials, building intuition for calibration

Before reaching for an automated tool, let's build intuition. The residual map above shows systematic patterns — what if we simply adjusted K?

We'll try three values: K = 150, 200, 300 m/d (i.e. multipliers of 0.75, 1.0, 1.5 on the baseline).

In [ ]:
# --- Quick manual trials: test three uniform K values ---
trial_multipliers = [0.75, 1.0, 1.5]
trial_k_values = [200.0 * m for m in trial_multipliers]

for k_val, mult in zip(trial_k_values, trial_multipliers):
    # Set K directly (uniform field)
    gwf.npf.k.set_data(k_val)
    sim.write_simulation()
    success_trial, _ = sim.run_simulation(silent=True)
    if not success_trial:
        print(f"  K = {k_val:.0f} m/d  →  model FAILED")
        continue

    head_trial = gwf.output.head().get_data()
    sim_heads_trial = cu.extract_heads_at_observations(head_trial, obs_gdf, modelgrid)
    valid = ~np.isnan(sim_heads_trial)
    metrics_trial = cu.calculate_calibration_metrics(
        obs_gdf.loc[valid, 'head_m'].values, sim_heads_trial[valid],
    )
    print(f"  K = {k_val:5.1f} m/d  (×{mult})  →  RMSE = {metrics_trial['RMSE']:.2f} m")

# Restore baseline K = 200 for subsequent sections
gwf.npf.k.set_data(200.0)
sim.write_simulation()
_ = sim.run_simulation(silent=True)
print("\n(K restored to 200 m/d for the rest of the notebook)")

In [ ]:
# Checkpoint: which K multiplier gave the best fit?
create_multiple_choice('task05_checkpoint_manual')

### 2.3 - Discussion

With only a few real observation points (all in a single cluster) we could adjust K to fit them reasonably well. But we have very limited information about the rest of the domain. The synthetic observations help us assess model performance more broadly, but they are derived from the model itself.

This motivates two actions:
1. **Get more data** — perform a pumping test in a data-sparse area
2. **Allow K to vary spatially** — because the pumping test may reveal different K values

---

## 3 - Pumping Test Evaluation

To get an independent estimate of K in the central part of the aquifer - where we have no observation wells — we performed a **pumping test**. A well was pumped at a constant rate while drawdown was monitored at 4 observation wells at different distances.

We analyse this test using the **Cooper-Jacob straight-line method**. The maximum drawdown remains below 9 % of the aquifer thickness, so the confined-aquifer approximation underlying Cooper-Jacob is acceptable.

### 3.1 Cooper-Jacob Theory Reminder

<details>
<summary><strong>Click to expand: Cooper-Jacob equations</strong></summary>

The **Theis solution** describes drawdown around a pumping well in a confined, infinite aquifer:

$$s(r,t) = \frac{Q}{4\pi T}\, W(u) \qquad \text{where} \quad u = \frac{r^2 S}{4Tt}$$

For **late times** when $u < 0.01$, the Cooper-Jacob approximation gives a **straight line** on a semi-log plot ($s$ vs. $\log_{10} t$):

$$s \approx \frac{2.3\, Q}{4\pi T} \log_{10}\left(\frac{2.25\, T\, t}{r^2 S}\right)$$

From the slope $\Delta s$ (drawdown per log cycle) and the zero-intercept $t_0$:

$$T = \frac{2.3\, Q}{4\pi\, \Delta s} \qquad S = \frac{2.25\, T\, t_0}{r^2} \qquad K = \frac{T}{b}$$

**Key insight:** The slope depends only on $Q$ and $T$, not on distance $r$. All observation wells should yield the **same $T$**.

</details>

### 3.2 Assumptions and applicability

The Cooper-Jacob method relies on several assumptions. For our unconfined Limmat Valley aquifer, we should check each one:

| Assumption | Meaning | Our test |
|:--|:--|:--|
| **Confined aquifer** | Constant transmissivity (no dewatering) | Unconfined — but max drawdown < 9 % of *b* |
| **Homogeneous & isotropic** | K constant within the cone of depression | Reasonable at the test scale (~100 m) |
| **Fully penetrating well** | Screen spans the full aquifer thickness | Assumed |
| **Infinite lateral extent** | No boundaries within cone of influence | OK for a 24 h test |
| **Constant Q** | Steady pumping rate | Yes |
| **Late time (u < 0.01)** | Only late data used for the straight line | Checked per well |

**Unconfined correction (Jacob, 1963):** When the aquifer is unconfined, drawdown reduces the saturated thickness and therefore the effective transmissivity. The standard fix is to replace measured drawdown $s$ with a **corrected** value:

$$s' = s - \frac{s^2}{2b}$$

where $b$ is the pre-pumping saturated thickness. You then apply Cooper-Jacob to $s'$ instead of $s$. The correction is negligible when $s \ll b$ — we'll check this quantitatively after the analysis.

### 3.3 Load pumping test data

In [ ]:
# --- 1. Generate pumping test data ---
# (Reproducible synthetic data from the Theis solution + measurement noise)
from generate_pumping_test_data import generate_pumping_test_data

pt_output_dir = os.path.join(workspace, 'pumping_test')
os.makedirs(pt_output_dir, exist_ok=True)
csv_path, json_path = generate_pumping_test_data(output_dir=pt_output_dir)

# Load the data
pt_data = pd.read_csv(csv_path)
with open(json_path) as f:
    pt_meta = json.load(f)

# Given information
Q = pt_meta['pumping_rate_m3_per_day']   # pumping rate [m³/d]
b = pt_meta['aquifer_thickness_m']        # aquifer thickness [m]

print(f"Pumping test: {len(pt_data)} measurements, "
      f"{pt_data['well_id'].nunique()} observation wells")
print(f"Q = {Q:.0f} m³/d ({Q/86.4:.1f} L/s),  b = {b:.0f} m")
print()
for wid, grp in pt_data.groupby('well_id'):
    r = grp['distance_m'].iloc[0]
    print(f"  {wid}: r = {r:.0f} m, {len(grp)} points")

### 3.4 Raw drawdown data

As seen in the next graph, closer wells (OW-1, OW-2) show larger drawdown and respond faster. Farther wells respond more slowly. All curves approach straight lines on a semi-log plot, thi is the **Cooper-Jacob regime**.

In [ ]:
fig, ax = ptu.plot_all_wells_raw(pt_data)
plt.show()

### 3.5 Cooper-Jacob Analysis — Step by Step exercise

✏️ **Exercise:** Work through the Cooper-Jacob method step by step, starting with observation well **OW-2** (r = 30 m).

#### 3.5.1 - Step 1: Fit the straight line**

To fit the late-time straight line and to visualise the fit, use the two following functions: 

```python
ptu.cooper_jacob_fit(time_min, drawdown_m)          → dict with 'slope', 't0_min', ...
ptu.plot_semilog(time, drawdown, fit, well_id, r_m) → ax
```

This enables you to read off the **slope** $\Delta s$ (drawdown per log cycle).

In [ ]:
# --- Step 1: Cooper-Jacob fit for OW-2 ---

# Extract OW-2 data (provided)
ow2 = pt_data[pt_data['well_id'] == 'OW-2']
r_ow2 = ow2['distance_m'].iloc[0]

# ✏️ Fit the Cooper-Jacob straight line to OW-2 drawdown data
fit_ow2 = ptu.cooper_jacob_fit(
    ow2['time_min'].values,  # ← time array (hint: ow2['time_min'].values)
    ow2['drawdown_m'].values,  # ← drawdown array (hint: ow2['drawdown_m'].values)
)

# ✏️ Plot the semi-log fit
ax = ptu.plot_semilog(
    ow2['time_min'].values,  # ← time array
    ow2['drawdown_m'].values,  # ← drawdown array
    fit_ow2,
    well_id='OW-2',
    r_m=r_ow2,
)
plt.show()

# Read off the fitted slope
delta_s = fit_ow2['slope']
print(f"\nΔs (slope per log cycle): {delta_s:.3f} m")
print(f"t₀ (zero-drawdown time):  {fit_ow2['t0_min']:.3f} min")

In [ ]:
# Checkpoint: verify your slope value
check_task_with_solution('task05_pt_checkpoint_1')

#### 3.5.2 - Step 2: Calculate transmissivity

Apply the Cooper-Jacob formula to compute transmissivity $T$ from the slope: $T = \frac{2.3\, Q}{4\pi\, \Delta s}$

**Available variables:** $Q$ (pumping rate in m³/d), $\Delta s$ (slope from Step 1)

In [ ]:
# --- Step 2: Calculate transmissivity T for OW-2 ---
# Formula: T = 2.3 * Q / (4 * π * Δs)

           
T_ow2 =  2.3 * Q / (4 * 3.14 * delta_s)# ✏️ write the formula using Q and delta_s

print(f"T = 2.3 × {Q:.0f} / (4π × {delta_s:.3f}) = {T_ow2:.1f} m²/d")

In [ ]:
# Checkpoint: verify your T value
check_task_with_solution('task05_pt_checkpoint_2')

#### 3.5.3 - Step 3: Convert to hydraulic conductivity

We know that $K = \frac{T}{b}$.

Available variables:  $T$ (`T_ow2` from Step 2), $b$ (aquifer thickness in m)

In [ ]:
# --- Step 3: Convert T to hydraulic conductivity K ---
# Formula: K = T / b

K_ow2 = T_ow2 / b # ✏️ write the formula using T_ow2 and b

print(f"K = {T_ow2:.1f} / {b:.0f} = {K_ow2:.1f} m/d")
print(f"Compare with Notebook 4 uniform K = 200 m/d")

In [ ]:
# Checkpoint: verify your K value
check_task_with_solution('task05_pt_checkpoint_3')

#### 3.5.4 - Step 4: Extend to all wells

If the Theis assumptions hold, all 4 observation wells should give the **same $T$** — because the Cooper-Jacob slope depends only on $Q$ and $T$, not on distance $r$. Use `ptu.analyze_all_wells()` to run the analysis on all wells at once, then `ptu.summarize_results()` to print a comparison table.

```python
# Function signatures:
ptu.analyze_all_wells(df, Q_m3d=..., b_m=...)  → (results_df, fits_dict)
ptu.summarize_results(results_df)               → prints table
```

In [ ]:
# --- Step 4: Extend analysis to all 4 observation wells ---

# ✏️ Run Cooper-Jacob on all wells
results_df, fits = ptu.analyze_all_wells(
    pt_data,      # ← pumping test dataframe
    Q_m3d=3000,  # ← pumping rate [m³/d]
    b_m=b,    # ← aquifer thickness [m]
)

# ✏️ Print the results summary
ptu.summarize_results(results_df)  # ← pass the results dataframe

# Semi-log plots for all 4 wells (visual check)
fig, axes = ptu.plot_all_wells_cj(pt_data, fits, results_df)
fig.suptitle('Cooper-Jacob Analysis — All Observation Wells', fontsize=14, y=1.02)
plt.show()

# --- Store mean K for use in PEST++ calibration (Section 5) ---
K_mean = results_df['K_md'].mean()
K_pumping_test = K_mean

print(f"→ K from pumping test: {K_pumping_test:.1f} m/d (mean of {len(results_df)} wells)")

In [ ]:
# Checkpoint: verify mean K across all wells
check_task_with_solution('task05_pt_checkpoint_4')

#### 3.5.5 - Step 5: Interpret the results


In [ ]:
# Checkpoint: why does T vary slightly across wells?
create_multiple_choice('task05_pt_checkpoint_5')

### 3.6 Jacob correction for unconfined conditions and final interpretation

Our aquifer is unconfined, so drawdown reduces the saturated thickness. Let's apply the Jacob correction $s' = s - s^2/(2b)$ and re-run the Cooper-Jacob analysis to see how much it matters.

As shown right after, the consistency of $T$ across wells validates the analysis. The pumping test gives us $K \approx 300$ m/d in the central aquifer. This is higher than the uniform K = 200 m/d from Notebook 4, but consistent with coarse gravels in the central alluvial valley. The Jacob unconfined correction changes K by only a few percent, confirming that Cooper-Jacob is acceptable here.

> **Think about it:** If the PT-derived K differs from the uniform K value, what does that tell us about the spatial variability of K?

In [ ]:
# --- Jacob correction: s' = s - s²/(2b) ---
pt_data_corrected = pt_data.copy()
pt_data_corrected['drawdown_m'] = (
    pt_data['drawdown_m'] - pt_data['drawdown_m']**2 / (2 * b)
)

# Re-run Cooper-Jacob on corrected drawdowns
results_corrected, _ = ptu.analyze_all_wells(
    pt_data_corrected, Q_m3d=Q, b_m=b,
)

# Compare uncorrected vs corrected
print(f"{'Well':<8} {'T (m²/d)':>12} {'T_corr (m²/d)':>14} {'K (m/d)':>10} {'K_corr (m/d)':>13} {'ΔK (%)':>8}")
print("-" * 68)
for _, row_unc in results_df.iterrows():
    wid = row_unc['well_id']
    row_cor = results_corrected[results_corrected['well_id'] == wid].iloc[0]
    dk_pct = (row_cor['K_md'] - row_unc['K_md']) / row_unc['K_md'] * 100
    print(f"{wid:<8} {row_unc['T_m2d']:12.1f} {row_cor['T_m2d']:14.1f} "
          f"{row_unc['K_md']:10.1f} {row_cor['K_md']:13.1f} {dk_pct:8.1f}")

K_mean_corrected = results_corrected['K_md'].mean()
print(f"\nMean K (uncorrected): {K_mean:.1f} m/d")
print(f"Mean K (corrected):   {K_mean_corrected:.1f} m/d")
print(f"Difference:           {abs(K_mean_corrected - K_mean)/K_mean*100:.1f} %")
print(f"\nMax drawdown / aquifer thickness: {pt_data['drawdown_m'].max():.2f} / {b:.0f} m "
      f"= {pt_data['drawdown_m'].max()/b*100:.1f} %")
print("→ The correction is small because drawdown is small relative to aquifer thickness.")

---

## 4 - Calibration Concepts

Before running the calibration, let's review the key ideas.

### 4.1 Calibration metrics

| Metric | Formula | Interpretation | Target (for a water balance error < 1 %)|
|--------|---------|----------------|-----------------------------------------|
| **ME** | $\frac{1}{n}\sum(h_{sim} - h_{obs})$ | Bias (positive = over-prediction) | ≈ 0 |
| **MAE** | $\frac{1}{n}\sum\left\|h_{sim} - h_{obs}\right\|$ | Average error magnitude | |
| **RMSE** | $\sqrt{\frac{1}{n}\sum(h_{sim} - h_{obs})^2}$ | Penalises large errors | |
| **NRMSE** | $\frac{\text{RMSE}}{h_{max} - h_{min}} \times 100\%$ | RMSE relative to observed range | < 10 % |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | Fraction of variance explained | > 0.9 |


### 4.2 Pilot points, prior information, and PEST++

<details>
<summary><strong>Click to expand: Pilot points</strong></summary>

Instead of calibrating a few large K zones, we estimate K at **distributed pilot points**:
- Pilot points are scattered across the active domain
- Each has its own adjustable K parameter (log-transformed)
- Values between points are **interpolated** (kriging or IDW) to create a smooth K field
- More flexible than zones, but needs **regularisation** to avoid overfitting

**Preferred-value regularisation:** Each pilot point has a soft constraint pulling it toward the initial K estimate (200 m/d). Without this, pilot points far from any observation well are unconstrained and can drift to physically unrealistic values. The regularisation weight controls the trade-off: too low and you get spatial artefacts; too high and the calibration cannot adjust K enough to fit the data.

</details>

<details>
<summary><strong>Click to expand: Prior information</strong></summary>

The pumping test gives us $K \approx 300$ m/d at a specific location. We encode this as a **prior information equation** in PEST++. During calibration, PEST++ balances fitting head observations against honouring this independent estimate.

</details>

<details>
<summary><strong>Click to expand: PEST++ (pestpp-glm)</strong></summary>

**PEST++** is the industry standard for model-independent parameter estimation. We use **pestpp-glm** (Gauss-Levenberg-Marquardt):
1. Run the model with current parameters
2. Compute sensitivities (Jacobian matrix)
3. Update parameters to reduce the objective function
4. Repeat until convergence

</details>

### A note on observation weighting

In practice, observation weights reflect measurement confidence. Real AWEL wells (±0.1 m precision) would receive higher weight than synthetic points (±1.3 m noise). We use equal weights here for simplicity, but weighting is critical in production calibration, because it determines how much each observation influences the parameter estimates.

---

## 5 - OPTIONAL: PEST++ Calibration with Pilot Points

We set up and run PEST++ with:
- **~20 pilot points** distributed across the active domain
- **Observations:** AWEL mean-head measurements + synthetic observations
- **Prior information:** K from the pumping test
- **Parameter bounds:** 1–300 m/d (plausible for a gravel aquifer)
- **Preferred-value regularisation:** prevents unconstrained pilot points from drifting to extreme values

### 5.0 Choose calibration mode

In [ ]:
# === Calibration mode ===
# Set to False to download a pre-computed calibration
# Calibratio
# n can take a long time to run. Only set to True if you want to run 
# PEST++ locally and have it installed.
RUN_PEST_LOCALLY = False

### 5.1 Install PEST++ (if needed)

In [ ]:
# --- 1. Check if pestpp-glm is available; install if needed ---
from setup_pest_calibration import ensure_pestpp_installed

if RUN_PEST_LOCALLY:
    ensure_pestpp_installed()
else:
    print("Using pre-computed calibration; skipping PEST++ install.")

### 5.2 Build the PEST++ setup

In [ ]:
# --- 2. Import PEST setup utilities ---
from setup_pest_calibration import (
    build_pest_setup,
    run_pestpp_glm,
    load_pest_results,
    get_calibrated_k_field,
    read_pest_phi_progress,
    apply_calibrated_parameters,
    ensure_calibration_workspace,
    _interpolate_pp_to_grid,
    _nearest_pilot_point,
)

# --- 3. Place the pumping test in the central aquifer ---
xc = modelgrid.xcellcenters
yc = modelgrid.ycellcenters
active_mask = idomain.flatten() > 0

active_x = xc[active_mask]
active_y = yc[active_mask]
pt_x = float(np.median(active_x))
pt_y = float(np.median(active_y))
pt_location = (pt_x, pt_y)

# Fallback: if the pumping test exercise was skipped, use the derived K value
if 'K_pumping_test' not in dir():
    K_pumping_test = 300.0  # default — complete Section 3 to derive this yourself
    print("(Using default K = 300 m/d — complete the pumping test exercise to derive this yourself)")

print(f"Pumping test location: ({pt_x:.0f}, {pt_y:.0f})")
print(f"K from pumping test:   {K_pumping_test:.1f} m/d")

# --- 4. Build or download PEST++ calibration directory ---
if RUN_PEST_LOCALLY:
    # Identify which RIV cells belong to the Sihl (for leakance multiplier)
    from shapely.geometry import Point

    sihl_poly = river_gdf[river_gdf['GEWAESSERNAME'] == 'Sihl'].union_all()
    riv_data = riv.stress_period_data.get_data(0)

    sihl_cell_ids = []
    for rec in riv_data:
        cid = rec['cellid']
        cell_idx = cid[-1] if isinstance(cid, tuple) else cid
        pt = Point(xc[cell_idx], yc[cell_idx])
        if sihl_poly.contains(pt):
            sihl_cell_ids.append(cell_idx)
    sihl_cell_ids = np.array(sihl_cell_ids)
    print(f"Sihl RIV cells: {len(sihl_cell_ids)} out of {len(riv_data)} total")

    pest_ws, pp_xy = build_pest_setup(
        model_ws=workspace,
        modelgrid=modelgrid,
        obs_df=obs_gdf,
        pt_K=K_pumping_test,
        pt_location=pt_location,
        n_pilot_points=20,
        k_initial=200.0,
        k_lower=10.0,
        k_upper=600.0,
        pt_weight=1.5,
        reg_weight=0.5,
        variogram_range=600.0,
        idomain=idomain,
        sihl_cell_ids=sihl_cell_ids,
        anisotropy_angle=-30.0,
        anisotropy_scaling=3.0,
    )
else:
    pest_ws = ensure_calibration_workspace(DATA_DIR)
    pp_xy = np.load(os.path.join(pest_ws, 'pp_xy.npy'))   # v2: required by downstream 05f cells

# --- 5. Visualise pilot point locations ---
fig, ax = plt.subplots(figsize=(12, 8))

# Plot the active domain as background
active_arr = np.where(idomain.flatten() > 0, 1.0, np.nan)
quick_model_plot(modelgrid, active_arr, boundary_gdf=boundary_gdf,
                 cmap='Blues', colorbar_label='', ax=ax,
                 title='Pilot Points and Observation Network')

# Overlay pilot points and observation wells
ax.scatter(pp_xy[:, 0], pp_xy[:, 1], s=60, c='blue', marker='^',
           edgecolors='black', linewidth=0.8, zorder=4, label='Pilot points')

# Separate real and synthetic observations
real_obs = obs_gdf[~obs_gdf['is_synthetic']]
synth_obs = obs_gdf[obs_gdf['is_synthetic']]

if len(real_obs) > 0:
    ax.scatter(real_obs.geometry.x, real_obs.geometry.y,
               s=120, c='red', edgecolors='black', linewidth=1.2, zorder=5,
               label='AWEL wells')
if len(synth_obs) > 0:
    ax.scatter(synth_obs.geometry.x, synth_obs.geometry.y,
               s=80, c='#E67E22', marker='^', edgecolors='black', linewidth=1.0, zorder=5,
               label='Synthetic wells')

ax.scatter(*pt_location, s=200, c='gold', marker='*', edgecolors='black',
           linewidth=1.2, zorder=6, label='Pumping test site')

ax.legend(loc='lower left')
fig.tight_layout()
plt.show()

### 5.3 Run the calibration (pestpp-glm)
Running the calibration will take several minutes. The algorithm uses super parameters (SVD reduction) to minimise the number of model runs per iteration.

In [ ]:
# --- 6. Run PEST++ calibration ---
# This may take a few minutes depending on model size.
if RUN_PEST_LOCALLY:
    success = run_pestpp_glm(pest_ws)

    if success:
        print("\nCalibration completed successfully.")
    else:
        print("\nCalibration encountered issues — check the .rec file for details.")
else:
    print("Using downloaded pre-computed calibration.")

In [ ]:
# --- Convergence history ---
phi_df = read_pest_phi_progress(pest_ws)
if phi_df is not None and len(phi_df) > 1:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(phi_df['iteration'], phi_df['total_phi'], 'o-', label='Total phi')
    if 'meas_phi' in phi_df.columns:
        ax.plot(phi_df['iteration'], phi_df['meas_phi'], 's--', label='Measurement phi')
    if 'reg_phi' in phi_df.columns:
        ax.plot(phi_df['iteration'], phi_df['reg_phi'], '^--', alpha=0.6, label='Regularisation phi')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Objective Function (phi)')
    ax.set_title('PEST++ Convergence History')
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()
    print("Each iteration = one Jacobian computation + parameter update.")
    print("The objective function decreases as PEST++ improves the fit.")
else:
    print("Convergence history not available (no .iobj file found).")

### 5.4 Calibration results

#### 5.4.1 - View calibration results

In [ ]:
# --- 7. Load calibrated K field ---
try:
    k_field, pp_df = get_calibrated_k_field(
        pest_ws, modelgrid, variogram_range=600.0,
        anisotropy_angle=-30.0, anisotropy_scaling=3.0,
    )
    print("Calibrated K field loaded from PEST++ results.")
except Exception as e:
    print(f"Could not load PEST results ({e}). Using initial K field.")
    pp_log_vals = np.full(len(pp_xy), np.log10(20.0))
    k_field = _interpolate_pp_to_grid(
        pp_xy, pp_log_vals, modelgrid, method='idw', variogram_range=600.0,
    )
    pp_df = pd.DataFrame({
        'x': pp_xy[:, 0], 'y': pp_xy[:, 1],
        'K_md': np.full(len(pp_xy), 200.0),
    })

    # --- 8. Visualise calibrated K field ---
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Left panel: uniform K = 200 m/d (uncalibrated)
k_uniform = np.where(idomain.flatten() > 0, 200.0, np.nan)
k_cal_plot = np.where(idomain.flatten() > 0, k_field.flatten(), np.nan)
vmin_k = np.nanmin(k_cal_plot)
vmax_k = np.nanmax(k_cal_plot)

quick_model_plot(modelgrid, k_uniform, boundary_gdf=boundary_gdf,
                 cmap='viridis', colorbar_label='K [m/d]',
                 vmin=vmin_k, vmax=vmax_k,
                 title='Uncalibrated: Uniform K = 200 m/d', ax=axes[0])

# Right panel: calibrated K field with pilot points
quick_model_plot(modelgrid, k_cal_plot, boundary_gdf=boundary_gdf,
                 cmap='viridis', colorbar_label='K [m/d]',
                 vmin=vmin_k, vmax=vmax_k,
                 title='Calibrated: Spatially Varying K', ax=axes[1])

# Overlay pilot points colored by K
sc = axes[1].scatter(pp_df['x'], pp_df['y'], c=pp_df['K_md'],
                     cmap='viridis', s=50, edgecolors='black', linewidth=0.8,
                     vmin=vmin_k, vmax=vmax_k, zorder=4)
axes[1].scatter(*pt_location, s=200, c='gold', marker='*', edgecolors='black',
                linewidth=1.2, zorder=6, label=f'PT site (K\u2248{K_pumping_test:.0f} m/d)')
axes[1].legend(loc='lower right')

fig.tight_layout()
plt.show()



# --- 9. Apply calibrated K field and Sihl leakance multiplier ---
cal_result = apply_calibrated_parameters(
    gwf, pest_ws, modelgrid, river_gdf, boundary_gdf,
    variogram_range=600.0, anisotropy_angle=-30.0, anisotropy_scaling=3.0,
    set_npf_flags=False,
)
k_field = cal_result['k_field']
sihl_mult = cal_result['sihl_mult']

if sihl_mult != 1.0:
    print(f"Sihl leakance multiplier: {sihl_mult:.1f}\u00d7 "
          f"(effective \u03bb = {1.3e-7 * sihl_mult:.2e} 1/s)")

sim.write_simulation()
success_cal, _ = sim.run_simulation(silent=True)

if success_cal:
    head_cal = gwf.output.head().get_data()

    sim_heads_cal = cu.extract_heads_at_observations(head_cal, obs_gdf, modelgrid)
    obs_gdf['sim_head_calibrated'] = sim_heads_cal

    valid_cal = ~np.isnan(sim_heads_cal)
    obs_cal = obs_gdf[valid_cal].copy()

    metrics_calibrated = cu.calculate_calibration_metrics(
        obs_cal['head_m'].values, obs_cal['sim_head_calibrated'].values,
    )

    print(cu.compare_metrics(metrics_initial, metrics_calibrated,
                             labels=("Uncalibrated", "Calibrated")))
else:
    print("Calibrated model run failed.")
    head_cal = head

# Check for heads above land surface after calibration
if success_cal:
    top = disv.top.array.flatten()
    result_surface = check_heads_above_surface(
        head_cal, top, idomain, modelgrid, boundary_gdf=boundary_gdf
    )


In [ ]:
# Checkpoint 3: What is your calibrated RMSE (m)?
check_task_with_solution('task05_checkpoint_3')

In [ ]:

# --- 10. Before/after scatter plots ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

obs_cal = obs_gdf[~np.isnan(obs_gdf['simulated_head']) & ~np.isnan(obs_gdf.get('sim_head_calibrated', np.nan))].copy()

# Before calibration
ax = axes[0]
ax.scatter(obs_cal['head_m'], obs_cal['simulated_head'],
           c=['#E67E22' if s else '#2E86AB' for s in obs_cal['is_synthetic']],
           s=80, edgecolor='black', linewidth=0.5, zorder=3)
rng = [obs_cal['head_m'].min() - 1, obs_cal['head_m'].max() + 1]
ax.plot(rng, rng, 'k--', linewidth=1, zorder=1)
ax.set_xlim(rng); ax.set_ylim(rng)
ax.set_aspect('equal')
ax.set_xlabel('Observed Head (m a.s.l.)')
ax.set_ylabel('Simulated Head (m a.s.l.)')
ax.set_title('Before Calibration')
ax.grid(True, alpha=0.3)
ax.text(0.95, 0.05, f"RMSE = {metrics_initial['RMSE']:.2f} m\nR² = {metrics_initial['R2']:.3f}",
        transform=ax.transAxes, fontsize=10, va='bottom', ha='right',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# After calibration
if 'sim_head_calibrated' in obs_cal.columns:
    ax = axes[1]
    ax.scatter(obs_cal['head_m'], obs_cal['sim_head_calibrated'],
               c=['#E67E22' if s else '#2E86AB' for s in obs_cal['is_synthetic']],
               s=80, edgecolor='black', linewidth=0.5, zorder=3)
    ax.plot(rng, rng, 'k--', linewidth=1, zorder=1)
    ax.set_xlim(rng); ax.set_ylim(rng)
    ax.set_aspect('equal')
    ax.set_xlabel('Observed Head (m a.s.l.)')
    ax.set_ylabel('Simulated Head (m a.s.l.)')
    ax.set_title('After Calibration')
    ax.grid(True, alpha=0.3)
    ax.text(0.95, 0.05, f"RMSE = {metrics_calibrated['RMSE']:.2f} m\nR² = {metrics_calibrated['R2']:.3f}",
            transform=ax.transAxes, fontsize=10, va='bottom', ha='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

fig.tight_layout()
plt.show()

# --- Before/after head contour maps ---
fig, axes = plt.subplots(1, 3, figsize=(24, 7))

# Shared colour range across head maps
heads_uncal = np.where(idomain.flatten() > 0, head.flatten(), np.nan)
heads_cal_arr = np.where(idomain.flatten() > 0, head_cal.flatten(), np.nan)
vmin = np.nanmin([np.nanmin(heads_uncal), np.nanmin(heads_cal_arr)])
vmax = np.nanmax([np.nanmax(heads_uncal), np.nanmax(heads_cal_arr)])

for ax, arr, title in zip(axes[:2], [heads_uncal, heads_cal_arr],
                           ['Uncalibrated Heads', 'Calibrated Heads']):
    quick_model_plot(modelgrid, arr, boundary_gdf=boundary_gdf,
                     cmap='viridis', colorbar_label='Head (m a.s.l.)',
                     vmin=vmin, vmax=vmax, ax=ax, title=title)
    # Overlay observation wells
    real_obs = obs_gdf[~obs_gdf['is_synthetic']]
    synth_obs = obs_gdf[obs_gdf['is_synthetic']]
    if len(real_obs) > 0:
        ax.scatter(real_obs.geometry.x, real_obs.geometry.y,
                   s=60, c='red', edgecolors='black', linewidth=0.8, zorder=5)
    if len(synth_obs) > 0:
        ax.scatter(synth_obs.geometry.x, synth_obs.geometry.y,
                   s=40, c='#E67E22', marker='^', edgecolors='black', linewidth=0.8, zorder=5)

# Third panel: head difference (calibrated - uncalibrated)
head_diff = heads_cal_arr - heads_uncal
diff_abs = np.nanmax(np.abs(head_diff))
quick_model_plot(modelgrid, head_diff, boundary_gdf=boundary_gdf,
                 cmap='RdBu_r', colorbar_label='\u0394 Head (m)',
                 vmin=-diff_abs, vmax=diff_abs, ax=axes[2],
                 title='Difference (Calibrated \u2212 Uncalibrated)')
# Overlay observation wells on difference map too
if len(real_obs) > 0:
    axes[2].scatter(real_obs.geometry.x, real_obs.geometry.y,
                    s=60, c='red', edgecolors='black', linewidth=0.8, zorder=5)
if len(synth_obs) > 0:
    axes[2].scatter(synth_obs.geometry.x, synth_obs.geometry.y,
                    s=40, c='#E67E22', marker='^', edgecolors='black', linewidth=0.8, zorder=5)

fig.tight_layout()
plt.show()

#### 5.4.2 - Interpreting the calibration results: 

The calibrated model shows clear spatial variation in hydraulic conductivity, and the head difference map reveals where the heterogeneous K field shifts groundwater levels relative to the uniform-K baseline. The PEST++ calibration adjusts pilot-point K values to better match both real AWEL observations and synthetic targets, producing a physically plausible K field that varies across the valley. Areas near observation wells show the strongest constraint, while data-sparse regions retain more prior influence.

**📚 The Overfitting Trap:**
 When calibrating, more parameters don't always mean a better model. Consider the "U-curve" of model error:

 ```
 Error
  |╲                         ╱
  |  ╲    prediction error ╱
  |    ╲                 ╱
  |      ╲_____╱  ←  sweet spot
  |       ╱
  |     ╱  calibration error
  |   ╱
  | ╱
  └──────────────────────────── Parameters
 ```

 As you add parameters, calibration error always decreases — the model has more knobs to turn. But prediction error eventually *increases* because the model starts fitting noise rather than signal. This is **overfitting**. Regularisation (Section 4.2) and cross-validation (Notebook 6) are our defences against it. See also the FH-DGGV Leitfaden KalPro, Section 3.3.

### 5.5 Prior information check

**Why doesn't the PT K match exactly?** The next information equation pulls the nearest pilot point toward the PT value, but it competes with the head observations. PEST++ finds a compromise: if nearby observations require a different K to fit the heads, the PT prior can be overridden — especially when observation weights collectively outweigh the single prior constraint. This illustrates a fundamental trade-off in inverse modelling between **data fit** and **parameter plausibility**.

In [ ]:
# --- 11. Check PT K match ---
pp_pt_idx = _nearest_pilot_point(pp_xy, pt_location)
K_at_pt = pp_df.iloc[pp_pt_idx]['K_md'] if 'K_md' in pp_df.columns else 200.0

print(f"K from pumping test:              {K_pumping_test:.1f} m/d")
print(f"Calibrated K at nearest PP ({pp_pt_idx:02d}):  {K_at_pt:.1f} m/d")
print(f"Difference:                       {abs(K_at_pt - K_pumping_test):.1f} m/d "
      f"({abs(K_at_pt - K_pumping_test)/K_pumping_test*100:.0f}%)")

# Sihl leakance multiplier summary
if sihl_mult != 1.0:
    print(f"\nSihl leakance multiplier:         {sihl_mult:.1f}\u00d7")
    print(f"Effective Sihl \u03bb:                {1.3e-7 * sihl_mult:.2e} 1/s "
          f"(base: 1.3e-7 1/s)")

---

## 6 - Calibration Assessment

### 6.1 Water balance verification

In [ ]:
# --- 12. Water balance ---
try:
    budget_df = format_budget_summary(gwf, sim)
    display(budget_df)

    # Check water balance error
    pct_error = budget_df.loc['PERCENT ERROR', 'Net (m3/d)']
    if isinstance(pct_error, (int, float)) and pct_error < 1.0:
        print("\n  Water balance error < 1 % — PASS")
    elif isinstance(pct_error, (int, float)):
        print(f"\n  Water balance error = {pct_error:.4f} % — review model setup")
except Exception as e:
    print(f"Could not read water balance: {e}")

In [ ]:
# Checkpoint 4: What is the water balance error (%)?
check_task_with_solution('task05_checkpoint_4')

### 6.2 Residual map

A map and a histogram of calibration residuals are presented, and help assess whether errors are randomly distributed (desirable) or systematically biased. Ideally, residuals on the histogramm should be approximately centred on zero with no heavy tails.


<details>
<summary>Why is one pumping-test estimate of K particularly valuable when we already have head observations at four wells?</summary>

The four AWEL wells are all clustered in one area and constrain the model only locally. The pumping test provides an **independent, physically-based** measurement of K at a different location — this breaks the spatial non-uniqueness and prevents the calibration from assigning unrealistic K values where no head data exists.

</details>

In [ ]:
# --- 13. Residual map — calibrated model ---
if 'sim_head_calibrated' in obs_gdf.columns:
    obs_cal = obs_gdf[~np.isnan(obs_gdf['sim_head_calibrated'])].copy()
    residuals_cal = obs_cal['sim_head_calibrated'].values - obs_cal['head_m'].values
    fig = cu.plot_residual_map(
        obs_cal, residuals_cal,
        boundary_gdf=boundary_gdf,
        title='Residual Map — Calibrated Model',
    )
    plt.show()

    # --- Residual histogram ---
fig, ax = plt.subplots(figsize=(8, 4))
n_bins = min(len(residuals_cal), 10)
ax.hist(residuals_cal, bins=n_bins, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(x=0, color='red', linestyle='--', linewidth=1.5, label='Zero')
ax.axvline(x=np.mean(residuals_cal), color='orange', linestyle='-', linewidth=1.5,
           label=f'Mean = {np.mean(residuals_cal):.2f} m')
ax.set_xlabel('Residual (m)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Calibration Residuals')
ax.legend()
ax.text(0.95, 0.95, f'n = {len(residuals_cal)}\nσ = {np.std(residuals_cal):.2f} m',
        transform=ax.transAxes, va='top', ha='right',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
fig.tight_layout()
plt.show()

### 6.3 OPTIONAL - advanced: Non-uniqueness demonstration 

Can a different parameter combination produce a similar fit?

This advanced section requires a local calibration run.

Let's test the **K–recharge trade-off**: higher K with lower recharge vs. lower K with higher recharge.
Results are given in the next cell.

We notice **equifinality**: multiple parameter sets can fit the data similarly. The K–recharge trade-off arises because both parameters affect head levels: raising K lowers heads, but so does reducing recharge.
More observation data and independent measurements (like the pumping test, or river baseflow gauging) reduce non-uniqueness by constraining different aspects of the system.

In [ ]:
if RUN_PEST_LOCALLY:
    # --- Non-uniqueness: K–recharge trade-off ---
    # Reset to uniform K = 200 for clean trials
    gwf.npf.k.set_data(200.0)

    combos = [
        {"label": "High K + Low RCH", "k_global_multiplier": 1.3, "rch_multiplier": 0.8},
        {"label": "Low K + High RCH",  "k_global_multiplier": 0.8, "rch_multiplier": 1.3},
        {"label": "Baseline (K=40, RCH=1×)", "k_global_multiplier": 1.0, "rch_multiplier": 1.0},
    ]

    print("K–Recharge trade-off test:")
    print("-" * 60)
    for combo in combos:
        head_trial = cu.run_calibration_trial(
            sim, sim_name,
            k_global_multiplier=combo["k_global_multiplier"],
            rch_multiplier=combo["rch_multiplier"],
        )
        if head_trial is not None:
            sim_h = cu.extract_heads_at_observations(head_trial, obs_gdf, modelgrid)
            valid = ~np.isnan(sim_h)
            m = cu.calculate_calibration_metrics(
                obs_gdf.loc[valid, 'head_m'].values, sim_h[valid],
            )
            k_eff = 200.0 * combo["k_global_multiplier"]
            rch_mult = combo["rch_multiplier"]
            print(f"  {combo['label']:30s}  K={k_eff:5.1f} m/d, RCH×{rch_mult:.1f}  →  RMSE = {m['RMSE']:.2f} m")

    # Restore calibrated K field for subsequent cells
    gwf.npf.k.set_data(k_field)
    sim.write_simulation()
    _ = sim.run_simulation(silent=True)
else:
    print("Skipping non-uniqueness demonstration; it requires a local calibration run.")

In [ ]:
if RUN_PEST_LOCALLY:
    # Checkpoint (*Optional*): what additional data would most reduce K–recharge non-uniqueness?
    create_multiple_choice('task05_checkpoint_nonunique')
else:
    print("[RUN_PEST_LOCALLY is False — non-uniqueness checkpoint skipped.]")

---

## Summary

> **What we accomplished**
>
> - **Assessed** our observation network and found it severely limited (few real wells, one cluster)
> - **Supplemented** with synthetic observations for teaching purposes
> - **Performed** Cooper-Jacob analysis on a pumping test → independent K ≈ 300 m/d
> - **Calibrated** the model with PEST++ using pilot points and prior information
> - **Verified** the calibrated model through metrics comparison and water balance

**Key insight:** Calibration is only as good as the data that constrains it. With a few head observations and 1 pumping test, the calibrated K field is *one plausible solution* — not *the* solution. More observation data would dramatically reduce non-uniqueness.

---

## What You're Taking Forward

You now have a calibrated MODFLOW 6 model with:
- A spatially varying K field (from pilot-point calibration)
- Documented calibration metrics and water balance
- An understanding of the **limitations** and **non-uniqueness** inherent in this calibration

The biggest remaining question: **how reliable are the model's predictions?**

---

## Next Steps

**Continue to:** [06f_validation.ipynb](06f_validation.ipynb) — where we test the calibrated model against independent data not used in calibration.

After that: [07f_sensitivity_uncertainty.ipynb](07f_sensitivity_uncertainty.ipynb) — where we explore how parameter uncertainty propagates to predictions.

---

# Exercise 05f-1 - Pumping test

A transient pumping test with constant pumping rate $Q=31.6$ L/s was performed in a confined aquifer. 
In an observation well at distance $r=122$ m from the well, the drawdown values are measured (see the table below).


In [ ]:
sys.path.append('../../_SUPPORT/src/scripts/scripts_exercises')
from shared_functions import check_task_with_solution
import print_images as du

du.display_image(
    image_filename='PumpingTestTable.png',
    image_folder='3_exercises', 
    )
check_task_with_solution("pumping_test_1")
check_task_with_solution("pumping_test_2")

---
# References

- Anderson, M.P., Woessner, W.W., Hunt, R.J. (2015). *Applied Groundwater Modeling*. Academic Press.
- Cooper, H.H., Jacob, C.E. (1946). A generalized graphical method for evaluating formation constants and summarizing well-field history. *Trans. AGU*, 27(4), 526–534.
- Doherty, J. (2015). *Calibration and Uncertainty Analysis for Complex Environmental Models*. Watermark Numerical Computing.
- Fienen, M.N., Muffels, C.T., Hunt, R.J. (2009). On constraining pilot point calibration with regularization in PEST. *Ground Water*, 47(6), 835–844.
- White, J.T., et al. (2020). Towards improved environmental modeling outcomes. *Environ. Modelling & Software*, 131, 104749.